# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

***Objectifs du Notebook***
1.  **Ingestion de Données** : Centralisation des fichiers issus de **FBref** (statistiques techniques), **Transfermarkt** (données financières et biographiques) et du **Mapping WorldFootballR** (clé de correspondance).
2.  **Fusion Hybride Multi-Niveaux** : Mise en œuvre d'une stratégie de jointure en cascade pour maximiser le taux de matching :
    * *Niveau 1 & 2* : Utilisation du dictionnaire de mapping (Exact puis Fuzzy).
    * *Niveau 3 & 4* : Jointure directe par identité biographique (Nom + Année de naissance).


**Résultat attendu** : Une table finale exportable, prête pour l'analyse exploratoire et le Machine Learning, garantissant l'intégrité entre les performances sportives et la valorisation économique.

In [2]:
# Importation des packages nécessaires

import pandas as pd
import unicodedata
import re
from rapidfuzz import process, fuzz

## 2. Fonctions de normalisation des patronymes
Mise en place du pipeline de nettoyage :
* **Fix Encoding** : Correction des erreurs d'encodage (ex: Latin-1 vs UTF-8) pour stabiliser les caractères spéciaux.
* **Normalize Name** : Passage en minuscules, suppression des accents (ASCII folding) et retrait des caractères spéciaux par regex.
* **Remove Matched** : Fonction utilitaire permettant de filtrer dynamiquement le DataFrame des joueurs restants afin d'éviter les doublons lors des phases de fusion successives.

In [3]:
# Fonctions nécessaires pour la synchronisation des noms FBref et ceux du mapping

def fix_encoding(name):
    if not isinstance(name, str): return ""
    try:
        return name.encode('raw_unicode_escape').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return name

def normalize_name(name):
    if not isinstance(name, str): return ""
    normalized = unicodedata.normalize('NFD', fix_encoding(name))
    ascii_name = normalized.encode('ascii', 'ignore').decode("utf-8").lower().strip()
    ascii_name = re.sub(r'[^a-z0-9 ]', ' ', ascii_name)
    return re.sub(r'\s+', ' ', ascii_name).strip()


def remove_matched(df, keys_matched):
    return df[~df['join_key'].isin(keys_matched)].copy()

## 3. Chargement et préparation des sources
Récupération des trois sources de données :
1. **Mapping** : Dictionnaire de correspondance FBref/Transfermarkt.
2. **FBref** : Données de performance (saison 2025-2026).
3. **Transfermarkt** : Valeurs de marché et données biographiques (DOB).
Création des clés pivots `join_key` et `dob_key` pour faciliter les jointures.

In [4]:
# Chargement des données
df_mapping = pd.read_csv("../data_finale/mapping_fbref_tm.csv", encoding='latin1')
df_fbref   = pd.read_csv("../data/fbref_datasets/players_data-2025_2026.csv")
df_tm      = pd.read_csv("../data/transfermarkt_datasets/players.csv")

# Normalisation des noms et des clés de jointure
df_mapping['PlayerFBref'] = df_mapping['PlayerFBref'].apply(fix_encoding)
df_mapping['join_key']    = df_mapping['PlayerFBref'].apply(normalize_name)

df_fbref['join_key'] = df_fbref['Player'].apply(normalize_name)
df_fbref['dob_key']  = df_fbref['Born'].astype(str).str.strip()

df_tm['join_key']      = (df_tm['first_name'].apply(normalize_name) + ' ' + df_tm['last_name'].apply(normalize_name)).str.strip()
df_tm['join_key_full'] = df_tm['name'].apply(normalize_name)
# On extrait uniquement l'année depuis date_of_birth TM pour matcher avec Born FBref
df_tm['dob_key']       = pd.to_datetime(df_tm['date_of_birth'], errors='coerce').dt.strftime('%Y')

results   = []
remaining = df_fbref.copy()

## 4. Pipeline de fusion multi-niveaux
Application d'une stratégie de repli en 4 étapes pour maximiser le taux de correspondance :
* **Niveau 1** : Jointure exacte via le dictionnaire de mapping.
* **Niveau 2** : Recherche floue (Fuzzy) sur le mapping (seuil > 90%).
* **Niveau 3** : Jointure directe FBref ↔ TM via le nom exact et l'année de naissance.
* **Niveau 4** : Recherche floue combinée à l'année de naissance pour les cas complexes.

In [5]:
# Etape 1

# On tente de lier les joueurs FBref au mapping worldfootballR par le nom nettoyé.
merge_1 = pd.merge(remaining, df_mapping[['join_key', 'tm_id']], on='join_key', how='inner')
merge_1['match_method'] = 'exact_name_mapping'
results.append(merge_1)

# On retire les joueurs trouvés de la liste 'remaining' pour ne pas les traiter deux fois
remaining = remove_matched(remaining, set(merge_1['join_key']))
print(f"[1] Nom exact (mapping)     : {len(merge_1):>5} | restants : {len(remaining)}")

# Niveau 2

# Pour les joueurs restants, on cherche des noms proches dans le mapping (ex: Mbappé vs Mbappé Lottin).
SCORE_MIN = 90
mapping_keys = df_mapping['join_key'].tolist()
fuzzy_rows = []

for _, row in remaining.iterrows():
    # extractOne trouve la correspondance la plus proche avec un score de similarité
    res = process.extractOne(row['join_key'], mapping_keys, scorer=fuzz.token_sort_ratio)
    if res and res[1] >= SCORE_MIN:
        tm_id = df_mapping[df_mapping['join_key'] == res[0]].iloc[0]['tm_id']
        fuzzy_rows.append({**row.to_dict(), 'tm_id': tm_id, 'match_method': f'fuzzy_name({res[1]})'})

merge_2 = pd.DataFrame(fuzzy_rows)
if not merge_2.empty:
    results.append(merge_2)
    remaining = remove_matched(remaining, set(merge_2['join_key']))
print(f"[2] Fuzzy nom (mapping)     : {len(merge_2):>5} | restants : {len(remaining)}")


# Niveau 3 : jointure directe FBref - Transfermarkt

# Si le mapping échoue, on tente de lier directement FBref à Transfermarkt via le nom ET l'année de naissance.
tm_slim = df_tm[['player_id', 'join_key', 'dob_key']].rename(columns={'player_id': 'tm_id'})
merge_3a = pd.merge(remaining, tm_slim, on=['join_key', 'dob_key'], how='inner')

# On teste aussi sur le 'nom complet' de Transfermarkt au cas où
tm_slim_full = df_tm[['player_id', 'join_key_full', 'dob_key']].rename(
    columns={'player_id': 'tm_id', 'join_key_full': 'join_key'}
)
merge_3b = pd.merge(remaining, tm_slim_full, on=['join_key', 'dob_key'], how='inner')

# Fusion des deux tentatives (a et b) et suppression des doublons
merge_3 = pd.concat([merge_3a, merge_3b]).drop_duplicates(subset='join_key')
merge_3['match_method'] = 'exact_name+dob'
results.append(merge_3)
remaining = remove_matched(remaining, set(merge_3['join_key']))
print(f"[3] Nom exact + DOB (TM)    : {len(merge_3):>5} | restants : {len(remaining)}")


# Niveau 4

# Cas le plus complexe : on matche les joueurs nés la même année, puis on compare leurs noms.
tm_dob = df_tm[['player_id', 'join_key', 'join_key_full', 'dob_key']].rename(columns={'player_id': 'tm_id'})
# On crée des paires de candidats basées uniquement sur l'année de naissance
candidates = pd.merge(remaining, tm_dob, on='dob_key', how='inner', suffixes=('_fbref', '_tm'))

fuzzy_dob_rows = []
for _, row in candidates.iterrows():
    # Calcul de similarité entre le nom FBref et les deux variantes de noms TM
    score_1 = fuzz.token_sort_ratio(row['join_key_fbref'], row['join_key_tm'])
    score_2 = fuzz.token_sort_ratio(row['join_key_fbref'], row['join_key_full'])
    best_score = max(score_1, score_2)
    
    # Seuil de confiance à 80% car l'année de naissance (dob_key) valide déjà fortement l'identité
    if best_score >= 80:
        fuzzy_dob_rows.append({
            **{k: v for k, v in row.items() if k not in ['join_key_tm', 'join_key_full']},
            'join_key':     row['join_key_fbref'],
            'match_method': f'dob+fuzzy({best_score})'
        })

merge_4 = pd.DataFrame(fuzzy_dob_rows)
if not merge_4.empty:
    merge_4 = merge_4.drop_duplicates(subset='join_key')
    results.append(merge_4)
    remaining = remove_matched(remaining, set(merge_4['join_key']))
print(f"[4] DOB + fuzzy nom (TM)    : {len(merge_4):>5} | restants : {len(remaining)}")


# Assemblage final

# Extraction des colonnes de base pour l'assemblage
cols_base = [c for c in results[0].columns if c in df_fbref.columns or c in ['tm_id', 'match_method']]

# Concaténation de tous les niveaux de résultats (1, 2, 3 et 4)
df_with_id = pd.concat(
    [r[[c for c in cols_base if c in r.columns]] for r in results],
    ignore_index=True
)

# Renommage des colonnes TM pour éviter les conflits de noms lors de la fusion finale
df_tm_final = df_tm.rename(columns={
    'join_key':      'tm_join_key',
    'join_key_full': 'tm_join_key_full',
    'dob_key':       'tm_dob_key'
})

# Fusion finale : on récupère toutes les colonnes de Transfermarkt (Valeur, Poste, Pied, etc.) via le tm_id
df_final = pd.merge(df_with_id, df_tm_final, left_on='tm_id', right_on='player_id', how='inner')


# Rapport
print(f"\nFusion terminée          : {len(df_final)} joueurs")

print(f"\nRépartition par méthode :")
print(df_final['match_method'].value_counts().to_string())

# Identification des échecs (ceux qui restent dans FBref mais ne sont pas dans df_final)
still_missing = df_fbref[~df_fbref['join_key'].isin(df_final['join_key'])]
if not still_missing.empty:
    print(f"\n{len(still_missing)} joueurs toujours non matchés :")
    print(still_missing[['Player', 'Born', 'Squad', 'join_key']].to_string(index=False))
else:
    print("\nAucun joueur manquant !")

[1] Nom exact (mapping)     :  2299 | restants : 557
[2] Fuzzy nom (mapping)     :    19 | restants : 538
[3] Nom exact + DOB (TM)    :     0 | restants : 538
[4] DOB + fuzzy nom (TM)    :     0 | restants : 538

Fusion terminée          : 2296 joueurs

Répartition par méthode :
match_method
exact_name_mapping               2278
fuzzy_name(96.55172413793103)       3
fuzzy_name(92.3076923076923)        2
fuzzy_name(96.0)                    2
fuzzy_name(96.2962962962963)        2
fuzzy_name(92.85714285714286)       2
fuzzy_name(95.65217391304348)       1
fuzzy_name(95.23809523809523)       1
fuzzy_name(93.75)                   1
fuzzy_name(97.14285714285714)       1
fuzzy_name(100.0)                   1
fuzzy_name(90.0)                    1
fuzzy_name(95.0)                    1

542 joueurs toujours non matchés :
                     Player   Born               Squad                            join_key
                Zach Abbott 2006.0   Nottingham Forest                         zach ab

In [6]:
df_final

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur,tm_join_key,tm_join_key_full,tm_dob_key
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,25.0,2000.0,32,26,...,9.0,NaN,https://www.transfermarkt.co.uk/brenden-aarons...,GB1,Leeds United Association Football Club,18000000.0,30000000.0,brenden aaronson,brenden aaronson,2000
1,4,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,26.0,1999.0,7,1,...,0.0,NaN,https://www.transfermarkt.co.uk/himad-abdelli/...,FR1,Olympique de Marseille,7000000.0,7000000.0,himad abdelli,himad abdelli,1999
2,5,Himad Abdelli,dz ALG,MF,Angers,fr Ligue 1,26.0,1999.0,13,11,...,0.0,NaN,https://www.transfermarkt.co.uk/himad-abdelli/...,FR1,Olympique de Marseille,7000000.0,7000000.0,himad abdelli,himad abdelli,1999
3,6,Ali Abdi,tn TUN,"MF,DF",Nice,fr Ligue 1,32.0,1993.0,18,12,...,7.0,NaN,https://www.transfermarkt.co.uk/ali-abdi/profi...,FR1,Olympique Gymnaste Club Nice Côte d'Azur,2500000.0,3000000.0,ali abdi,ali abdi,1993
4,7,Salis Abdul Samed,gh GHA,MF,Nice,fr Ligue 1,26.0,2000.0,15,10,...,0.0,NaN,https://www.transfermarkt.co.uk/salis-abdul-sa...,FR1,Olympique Gymnaste Club Nice Côte d'Azur,4500000.0,18000000.0,salis abdul samed,salis abdul samed,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2291,1903,Rafel Obrador,es ESP,MF,Torino,it Serie A,22.0,2004.0,11,9,...,1.0,NaN,https://www.transfermarkt.co.uk/rafa-obrador/p...,IT1,Torino Calcio,4000000.0,4000000.0,rafa obrador,rafa obrador,2004
2292,1952,Abdoul Ouattara,fr FRA,"MF,DF",Strasbourg,fr Ligue 1,20.0,2005.0,23,15,...,NaN,NaN,https://www.transfermarkt.co.uk/abou-ouattara/...,PO1,Vitória Sport Clube,400000.0,800000.0,abou ouattara,abou ouattara,1999
2293,2152,Lasse Rieß,de GER,GK,Mainz 05,de Bundesliga,24.0,2001.0,4,3,...,NaN,NaN,https://www.transfermarkt.co.uk/lasse-riess/pr...,L1,1. Fußball- und Sportverein Mainz 05,500000.0,700000.0,lasse rie,lasse rie,2001
2294,2259,Álex Sánchez,es ESP,FW,Elche,es La Liga,20.0,2006.0,1,0,...,51.0,NaN,https://www.transfermarkt.co.uk/alexis-sanchez...,ES1,Sevilla Fútbol Club S.A.D.,1400000.0,70000000.0,alexis sanchez,alexis sanchez,1988
